# Robot Maze Demo

A tiny simulation playground for the next robot challenge: do not run into walls, build a 2-D occupancy map, then solve a maze through cells the robot knows are free.

The robot gets perfect 8-ray proximity scans here. That is intentionally generous: this notebook is for seeing the control/data loop clearly before making the sensors messier.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd()
while root != root.parent and not (root / "golem2").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from golem2.experiments.maze_navigation_demo import run_demo

result = run_demo()
print("unsafe probe:", result["unsafe_probe"]["event"])
print("map counts:", result["counts"])
print("path length:", len(result["path"]), "cells")
print("path crosses occupied:", result["path_crosses_occupied"])

## 1. The Safety Reflex

First, the robot tries an unsafe forward motion near a wall. The simulator clips the command before the body would touch the wall. This is the behavior we want before any clever model is allowed to matter.

In [ ]:
probe = result["unsafe_probe"]
for key in ["requested", "applied", "nearest_front", "event"]:
    print(f"{key:>14}: {probe[key]}")
print("\nroute events:")
for key, value in result["events"].items():
    print(f"{key:>14}: {value}")

## 2. Maze And Driven Route

The blue trace is the route the simulated robot actually drove while gathering scans. The orange segment is the clipped unsafe probe: it asked for more, got less, and stayed out of the wall.

In [ ]:
walls = result["world"]["walls"]
steps = result["steps"]
start = result["world"]["start"]
goal = result["world"]["goal"]

xs = [start["x"]] + [step["after"]["x"] for step in steps]
ys = [start["y"]] + [step["after"]["y"] for step in steps]

fig, ax = plt.subplots(figsize=(7, 7))
for wall in walls:
    ax.plot([wall["x1"], wall["x2"]], [wall["y1"], wall["y2"]], color="black", linewidth=3)
ax.plot(xs, ys, color="#2f6fdb", linewidth=2.5, label="driven scan route")
ax.scatter([start["x"]], [start["y"]], s=100, color="#2f6fdb", label="start")
ax.scatter([goal["x"]], [goal["y"]], s=120, color="#26a269", marker="*", label="goal")

before = probe["before"]
after = probe["after"]
ax.arrow(before["x"], before["y"], after["x"] - before["x"], after["y"] - before["y"],
         width=0.015, head_width=0.08, color="#e66100", length_includes_head=True,
         label="clipped unsafe probe")

ax.set_aspect("equal")
ax.set_xlim(-2.2, 2.2)
ax.set_ylim(-2.2, 2.2)
ax.set_title("Simulated maze, safe motion, and scan route")
ax.legend(loc="upper left")
ax.grid(alpha=0.2)
plt.show()

## 3. Occupancy Map And Solved Path

The robot exports three map states: unknown, free, and occupied. The planner is conservative: it only walks through known free cells, plus the declared goal cell.

In [ ]:
grid = result["grid"]
path = result["path"]

cmap = ListedColormap(["#2f3037", "#f6f0df", "#cb3b3b"])
display_grid = [[value + 1 for value in row] for row in grid]

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(display_grid, origin="lower", cmap=cmap, interpolation="nearest")
if path:
    px = [cell[0] for cell in path]
    py = [cell[1] for cell in path]
    ax.plot(px, py, color="#1c71d8", linewidth=2.5, label="A* over known free cells")
    ax.scatter([px[0]], [py[0]], color="#1c71d8", s=90, label="start cell")
    ax.scatter([px[-1]], [py[-1]], color="#26a269", marker="*", s=140, label="goal cell")
ax.set_title("Occupancy map: unknown / free / occupied with solved path")
ax.set_xticks([])
ax.set_yticks([])
ax.legend(loc="upper right")
plt.show()

print("free:", result["counts"]["free"], "occupied:", result["counts"]["occupied"], "unknown:", result["counts"]["unknown"])
print("path crosses occupied:", result["path_crosses_occupied"])

## 4. Stack Sensors On Sensors

The maze demo also exposes a little telemetry stack. Three child sensors report front clearance, map coverage, and path confidence. A parent sensor smooths and clips them with settable knobs: `alpha`, lower bound, upper bound, nudge up/down, grow spread, and shrink spread.

In [ ]:
stack = result["telemetry_stack"]
sensor_rows = []
command_rows = []
for record in stack:
    payload = record["payload"]
    if record["kind"] == "sensor_control":
        command_rows.append({
            "command": payload["command"]["name"],
            "value": payload["command"]["value"],
            "alpha": payload["knobs"]["alpha"],
            "lower": payload["knobs"]["lower_bound"],
            "upper": payload["knobs"]["upper_bound"],
        })
    elif record["component"] == "maze_navigability_sensor":
        sensor_rows.append({
            "value": payload["value"],
            "raw": payload["raw_value"],
            "clipped": payload["clipped_value"],
            "alpha": payload["knobs"]["alpha"],
            "lower": payload["knobs"]["lower_bound"],
            "upper": payload["knobs"]["upper_bound"],
        })

for row in command_rows:
    print(row)

fig, ax = plt.subplots(figsize=(8, 3.5))
xs = list(range(len(sensor_rows)))
ax.plot(xs, [row["value"] for row in sensor_rows], marker="o", label="parent EMA value")
ax.plot(xs, [row["raw"] for row in sensor_rows], linestyle="--", label="raw child average")
ax.fill_between(xs, [row["lower"] for row in sensor_rows], [row["upper"] for row in sensor_rows], color="#9db7e8", alpha=0.25, label="active bounds")
ax.set_ylim(-0.15, 1.15)
ax.set_title("Stacked telemetry sensor under set/nudge/spread commands")
ax.legend(loc="lower right")
ax.grid(alpha=0.2)
plt.show()

## 5. Peek At The Robot's Memory

Each motion step remembers what was requested, what was actually applied, and why. Change `step_index` to inspect a different moment.

In [ ]:
step_index = 12
step = result["steps"][step_index]
print("step", step_index)
print("event:", step["event"])
print("requested:", round(step["requested"], 3), "applied:", round(step["applied"], 3))
print("pose before:", {k: round(v, 3) for k, v in step["before"].items()})
print("pose after: ", {k: round(v, 3) for k, v in step["after"].items()})